# BA3093 Teaching Demo: EV Charging Simulation and Risk Analysis

This notebook is a teaching-oriented walkthrough for the BA3093 group project.

It covers both technical streams:

- Option A: Monte Carlo simulation for profit-risk analysis.
- Option B: Queuing analysis (M/M/c baseline + M/G/c correction).

The notebook is structured to match report requirements:

1. Problem introduction
2. Data and parameters
3. Model design
4. Results and analysis
5. Sensitivity analysis
6. Conclusion and recommendations

## 1) Project directories, dependencies, and output conventions

We define standard folders for reproducibility:

- `data/`: source data
- `backend/`: model code
- `report_assets/`: exported figures
- `report/`: report outputs
- `notebooks/`: teaching notebooks

We also import required libraries for statistics, simulation, and visualization.

In [7]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional libraries (used if available)
try:
    import seaborn as sns
except Exception:
    sns = None

from scipy import stats

ROOT = Path.cwd()
if not (ROOT / "backend").exists() and (ROOT.parent / "backend").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
BACKEND_DIR = ROOT / "backend"
ASSET_DIR = ROOT / "report_assets"
REPORT_DIR = ROOT / "report"
NOTEBOOK_DIR = ROOT / "notebooks"

for d in [ASSET_DIR, REPORT_DIR, NOTEBOOK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BACKEND_DIR))

from data_processor import load_baseline_data
from queuing_model import QueuingSimulator
from monte_carlo import MonteCarloSimulator

plt.style.use("seaborn-v0_8-whitegrid")
print("ROOT:", ROOT)
print("Data exists:", DATA_DIR.exists())

## 2) Markdown report template, LaTeX formulas, and image embedding

The report uses Markdown + LaTeX:

- Inline: $\rho = \lambda / (c\mu)$
- Peak/off-peak split:

$$
\lambda_{peak}=3\lambda_{avg},\quad \lambda_{off}=0.6\lambda_{avg}
$$

- Queue correction:

$$
W_q^{M/G/c} \approx W_q^{M/M/c}\times\frac{1+CV^2}{2}
$$

Below we generate a minimal template in `report/report_template.md`.

In [ ]:
report_template = """# Group Report Template\n\n## Cover Page\n- Group Number: [Fill]\n- Members: [Fill]\n\n## Problem Introduction\n[Business problem and why simulation/queuing is appropriate.]\n\n## Model Formula\n$$\\rho = \\frac{\\lambda}{c\\mu}$$\n\n## Peak/Off-peak Method\n[Explain averaging bias and define $\\lambda_{peak}=3\\lambda_{avg}$, $\\lambda_{off}=0.6\\lambda_{avg}$.]\n\n## Figure Example\n![Queue Curve](../report_assets/fig_04_queue_sensitivity_curve.png)\n\n## References\n[APA format]\n"""

template_path = REPORT_DIR / "report_template.md"
template_path.write_text(report_template, encoding="utf-8")
print("Saved:", template_path)

## 3) Data loading, cleaning, and parameter table generation

We read core CSV files, check missing values, and build a parameter table directly reusable in the report.

In [9]:
df_inf = pd.read_csv(DATA_DIR / "inf.csv")
df_dur = pd.read_csv(DATA_DIR / "duration.csv")
df_occ = pd.read_csv(DATA_DIR / "occupancy.csv")
df_vol = pd.read_csv(DATA_DIR / "volume-11kW.csv")
df_ep = pd.read_csv(DATA_DIR / "e_price.csv")
df_sp = pd.read_csv(DATA_DIR / "s_price.csv")

print("Shapes:")
for name, df in [("inf", df_inf), ("duration", df_dur), ("occupancy", df_occ), ("volume", df_vol), ("e_price", df_ep), ("s_price", df_sp)]:
    print(name, df.shape)

print("\nMissing values:")
for name, df in [("inf", df_inf), ("duration", df_dur), ("occupancy", df_occ), ("volume", df_vol), ("e_price", df_ep), ("s_price", df_sp)]:
    print(name, int(df.isna().sum().sum()))

baseline = load_baseline_data()
parameter_table = pd.DataFrame([
    ["lambda", "Arrival rate", baseline["lambda_rate"], "sessions/hour/station", "Little's Law"],
    ["mu", "Service rate", baseline["mu"], "sessions/hour/pile", "1/mean service duration"],
    ["c", "Number of servers", baseline["c"], "chargers", "mean charge_count rounded"],
    ["CV", "Service-time coefficient of variation", baseline["service_time_cv"], "-", "std/mean of service durations"],
    ["c_dwell", "Dwell-time opportunity cost", baseline["dwell_cost_per_minute"], "RMB/(veh*min)", "managerial assumption (W = W_q + 1/\u03bc)"],
], columns=["Variable", "Meaning", "Value", "Unit", "Estimation / Assumption"])
parameter_table

## 4) Descriptive statistics and exploratory visualization

We compute key descriptive statistics and draw histograms / box plots / time-series profile.

In [10]:
# Service duration sample
_dur = pd.read_csv(DATA_DIR / "duration.csv", index_col=0)
_occ = pd.read_csv(DATA_DIR / "occupancy.csv", index_col=0)
common = _dur.columns.intersection(_occ.columns)
arr_d = _dur[common].to_numpy().flatten().astype(float)
arr_o = _occ[common].to_numpy().flatten().astype(float)
mask = (arr_o > 0.5) & (arr_d > 0) & np.isfinite(arr_d) & np.isfinite(arr_o)
service_duration = arr_d[mask] / arr_o[mask]
service_duration = service_duration[service_duration <= np.percentile(service_duration, 95)]

desc = pd.Series(service_duration).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
print(desc)
print("Skewness:", pd.Series(service_duration).skew())
print("Kurtosis:", pd.Series(service_duration).kurt())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].hist(df_inf["charge_count"], bins=30, color="#60a5fa", edgecolor="white")
axes[0].set_title("Charger count distribution")
axes[0].set_xlabel("chargers/station")

axes[1].hist(service_duration, bins=60, color="#34d399", edgecolor="white")
axes[1].set_title("Service duration distribution")
axes[1].set_xlabel("hours")

hourly_occ = _occ.copy()
hourly_occ.index = pd.to_datetime(hourly_occ.index)
hour_profile = hourly_occ.groupby(hourly_occ.index.hour).mean().mean(axis=1)
axes[2].plot(hour_profile.index, hour_profile.values, marker="o", color="#f59e0b")
axes[2].set_title("Hourly occupancy profile")
axes[2].set_xlabel("hour of day")

plt.tight_layout()
plt.show()

## 5) Input distribution fitting and parameter estimation

We compare candidate distributions for service duration and inspect goodness-of-fit statistics (informal teaching-level check).

In [11]:
# Fit exponential and gamma (Erlang-like)
exp_loc, exp_scale = stats.expon.fit(service_duration, floc=0)
g_shape, g_loc, g_scale = stats.gamma.fit(service_duration, floc=0)

ks_exp = stats.kstest(service_duration, "expon", args=(exp_loc, exp_scale))
ks_gamma = stats.kstest(service_duration, "gamma", args=(g_shape, g_loc, g_scale))

fit_df = pd.DataFrame([
    ["Exponential", exp_scale, np.nan, ks_exp.statistic, ks_exp.pvalue],
    ["Gamma/Erlang-like", g_scale, g_shape, ks_gamma.statistic, ks_gamma.pvalue],
], columns=["Distribution", "Scale", "Shape(k)", "KS statistic", "KS p-value"])
fit_df

## 6) Monte Carlo class design and call (Option A)

We define lightweight teaching wrappers (`InputDistribution`, `SimulationModel`, `ScenarioRunner`) and call project classes underneath.

In [ ]:
from dataclasses import dataclass

@dataclass
class InputDistribution:
    name: str
    params: dict

class SimulationModel:
    def __init__(self, baseline: dict):
        self.baseline = baseline

    def run(self, occupancy_change=0.0, service_fee_change=0.0, electricity_cost_change=0.0, dwell_cost_change=0.0, n_iter=1000):
        peak_lambda = self.baseline["peak_lambda_rate"] * (1.0 + occupancy_change)
        offpeak_lambda = self.baseline["offpeak_lambda_rate"] * (1.0 + occupancy_change)
        dwell_cost = self.baseline["dwell_cost_per_minute"] * (1.0 + dwell_cost_change)

        q_peak = QueuingSimulator(
            lam=peak_lambda,
            mu=self.baseline["mu"],
            c=self.baseline["c"],
            service_time_cv=self.baseline["service_time_cv"],
        ).compute()
        q_offpeak = QueuingSimulator(
            lam=offpeak_lambda,
            mu=self.baseline["mu"],
            c=self.baseline["c"],
            service_time_cv=self.baseline["service_time_cv"],
        ).compute()

        mc = MonteCarloSimulator(
            peak_lambda_rate=peak_lambda,
            offpeak_lambda_rate=offpeak_lambda,
            peak_hours=self.baseline["peak_hours"],
            offpeak_hours=self.baseline["offpeak_hours"],
            mu=self.baseline["mu"],
            c=self.baseline["c"],
            service_time_cv=self.baseline["service_time_cv"],
            mean_kwh=self.baseline["mean_kwh"],
            std_kwh=self.baseline["std_kwh"],
            mean_e_price=self.baseline["mean_e_price"],
            mean_s_price=self.baseline["mean_s_price"],
            std_e_price=self.baseline["std_e_price"],
            std_s_price=self.baseline["std_s_price"],
            wholesale_price=self.baseline["wholesale_price"],
            daily_fixed_cost=self.baseline["daily_fixed_cost"],
            dwell_cost_per_minute=dwell_cost,
            service_fee_change=service_fee_change,
            electricity_cost_change=electricity_cost_change,
            n_iter=n_iter,
            seed=42,
        ).run()

        return q_peak, q_offpeak, mc

class ScenarioRunner:
    def __init__(self, model: SimulationModel):
        self.model = model

    def baseline(self):
        return self.model.run(n_iter=1000)

model = SimulationModel(baseline)
runner = ScenarioRunner(model)
q_peak0, q_off0, mc0 = runner.baseline()
print(
    "Baseline peak util=", round(q_peak0.rho, 4),
    "peak queue wait(min)=", round(q_peak0.queue_waiting_time_minutes, 4),
    "peak dwell(min)=", round(q_peak0.dwell_time_minutes, 4),
    "mean profit=", round(mc0.mean_profit, 2),
    "mean dwell penalty=", round(mc0.mean_dwell_penalty, 2)
)

## 7) Queuing model class call (Option B)

We evaluate queue KPIs across demand shocks to examine stability and congestion behavior.

In [ ]:
queue_rows = []
for oc in [-0.3, 0.0, 0.5, 1.0, 2.0]:
    q_peak, q_off, mc = model.run(occupancy_change=oc, n_iter=1000)
    queue_rows.append([
        oc,
        q_peak.rho,
        q_peak.queue_waiting_time_minutes,
        q_off.queue_waiting_time_minutes,
        q_peak.dwell_time_minutes,
        q_off.dwell_time_minutes,
        mc.mean_peak_utilization,
        mc.mean_peak_wait_minutes,
    ])

queue_df = pd.DataFrame(queue_rows, columns=[
    "occupancy_change",
    "peak_utilization_rho",
    "peak_queue_wait_minutes",
    "offpeak_queue_wait_minutes",
    "peak_dwell_minutes",
    "offpeak_dwell_minutes",
    "mc_avg_peak_utilization",
    "mc_avg_peak_wait_minutes",
])
queue_df

## 7a) Queue waiting time vs. total dwell time

The queueing model produces two different time measures:

- **Queue waiting time** $W_q$: time spent waiting before charging starts.
- **Total dwell time** $W = W_q + 1/\mu$: total time a vehicle stays in the station, including both waiting and active charging.

In this project, the financial penalty is based on **total dwell time** rather than $W_q$ alone, because vehicles occupy station resources (parking space, charger capacity, and potential service opportunities) throughout the *entire* stay.

| Metric | Meaning | Used in profit penalty? |
|---|---|---|
| $W_q$ | Queue waiting time | No — used for service quality interpretation |
| $W = W_q + 1/\mu$ | Total dwell time (time-in-system) | **Yes** — used as station resource occupation cost |
| $c_{dwell}$ | Cost per vehicle-minute in station | **Yes** — applied to $W$ |

The cell below prints both $W_q$ and $W$ side-by-side to make the conscious modelling choice transparent.

In [ ]:
print("=== Queue waiting time (W_q) vs. Total dwell time (W) ===")
print(f"Peak   W_q = {q_peak0.queue_waiting_time_minutes:.4f} min  |  W = {q_peak0.dwell_time_minutes:.2f} min")
print(f"Off-peak W_q = {q_off0.queue_waiting_time_minutes:.4f} min  |  W = {q_off0.dwell_time_minutes:.2f} min")
print()
print("The profit model uses W (dwell_time_minutes) to compute the dwell-time penalty.")
print(f"Dwell-time penalty = {mc0.mean_dwell_penalty:.2f} RMB/day")
print(f"Dwell cost rate c_dwell = {baseline['dwell_cost_per_minute']:.3f} RMB/veh-min")

## 8) Risk metrics and steady-state indicators

For Monte Carlo: VaR, CVaR, probability of loss.

For queuing: queue waiting time, total dwell time, queue length, utilization, probability of waiting.

In [ ]:
profits = np.array(mc0.profits)
var5 = np.percentile(profits, 5)
cvar5 = profits[profits <= var5].mean()
loss_prob = (profits < 0).mean()

risk_table = pd.DataFrame([
    ["Expected profit", profits.mean()],
    ["Std of profit", profits.std()],
    ["VaR 5%", var5],
    ["CVaR 5%", cvar5],
    ["Probability of loss", loss_prob],
    ["Dwell-time penalty", mc0.mean_dwell_penalty],
    ["Peak utilization rho", q_peak0.rho],
    ["Peak queue wait (min)", q_peak0.queue_waiting_time_minutes],
    ["Off-peak queue wait (min)", q_off0.queue_waiting_time_minutes],
    ["Peak dwell time (min)", q_peak0.dwell_time_minutes],
    ["Off-peak dwell time (min)", q_off0.dwell_time_minutes],
    ["MC avg peak utilization", mc0.mean_peak_utilization],
    ["MC avg peak wait (min)", mc0.mean_peak_wait_minutes],
], columns=["Metric", "Value"])
risk_table

## 9) Sensitivity analysis and tornado chart

We perturb key inputs one at a time and compare expected-profit changes.

In [ ]:
scenarios = {
    "Service fee +20%": dict(service_fee_change=0.2),
    "Electricity cost +20%": dict(electricity_cost_change=0.2),
    "Occupancy +20%": dict(occupancy_change=0.2),
    "Dwell cost +100%": dict(dwell_cost_change=1.0),
}

base_mean = profits.mean()
delta = {}
for k, cfg in scenarios.items():
    q_peak, q_off, mc = model.run(
        occupancy_change=cfg.get("occupancy_change", 0.0),
        service_fee_change=cfg.get("service_fee_change", 0.0),
        electricity_cost_change=cfg.get("electricity_cost_change", 0.0),
        dwell_cost_change=cfg.get("dwell_cost_change", 0.0),
        n_iter=1000,
    )
    delta[k] = np.mean(mc.profits) - base_mean

labels = list(delta.keys())
values = np.array([delta[k] for k in labels])
order = np.argsort(np.abs(values))
labels = [labels[i] for i in order]
values = values[order]

plt.figure(figsize=(8, 4.6))
plt.barh(labels, values, color=["#16a34a" if v >= 0 else "#dc2626" for v in values])
plt.axvline(0, color="black", linewidth=1)
plt.title("Tornado-Style Sensitivity of Expected Profit")
plt.xlabel("Change in expected daily profit (RMB)")
plt.tight_layout()
plt.show()

## 10) Project flowchart generation and embedding

Mermaid flowchart for report/notebook embedding:

```mermaid
flowchart LR
    A[Input Data] --> B[Cleaning & Parameter Estimation]
    B --> C[Queue Model M/M/c]
    C --> D[M/G/c Correction]
    D --> E[Monte Carlo 1000+ Iterations]
    E --> F[Risk Metrics VaR/CVaR/Loss Prob]
    F --> G[Sensitivity & Recommendations]
```


## 11) visualization.py integration for report-ready figures

Run centralized plotting script and preview generated assets.

In [16]:
import subprocess

cmd = [str(ROOT / ".venv" / "bin" / "python"), str(ROOT / "visualization.py")]
print("Running:", " ".join(cmd))
_ = subprocess.run(cmd, cwd=str(ROOT), check=True, capture_output=True, text=True)
print(_.stdout.splitlines()[-3:])

assets = sorted([p.name for p in ASSET_DIR.glob("*.png")])
assets

## 12) Teaching summary and decision recommendations

### What this notebook demonstrates

- End-to-end data-to-decision pipeline.
- Option A and Option B integration.
- Peak/off-peak bimodal queue-risk coupling.
- Conscious modelling choice: dwell-time penalty vs. pure queue-waiting penalty.
- Report-ready figures and reproducible outputs.

### Managerial recommendations

1. Monitor peak-hour utilization and total dwell-time thresholds in real time.
2. Use dynamic pricing during peak windows rather than blind station expansion.
3. Trigger capacity expansion only when rolling peak congestion exceeds threshold.

### Limitations

- Hourly aggregated data limits event-level validation.
- Peak/off-peak split ratio (50%-50%) is a scenario assumption; future work should calibrate from hour-level occupancy data.
- TAZ-hour aggregated data is converted to representative single-station parameters, so results represent average-station risk rather than specific-site prediction.
- Future work can use discrete-event simulation with timestamp-level logs.
